# Smart Parking Dashboard
Ejecuta las celdas en orden. La última celda abre el dashboard en el navegador.

In [1]:
# !pip install dash plotly pandas numpy

In [2]:
!pip install jupyter-dash
!pip install dash==2.10.0

from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os

print('Librerías cargadas')

C:\Users\atelleriama\AppData\Local\anaconda3\Lib\site-packages\dash\dash.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, parse_version


Librerías cargadas


In [3]:
# ── RUTA AL CSV ───────────────────────────────────────────────────────────────
CSV_PATH = r"G:\Mi unidad\2 CURSO\Computer Science\trabajocomputer\parking_work\parking_data.csv"

TOTAL_SPACES = 10
GAS_LIMIT    = 450

In [4]:
# ── CARGAR CSV (datos reales) ─────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce")
df = df.dropna(subset=["Time"]).reset_index(drop=True)
print(f"CSV cargado: {len(df)} lecturas")
df.head()

CSV cargado: 1133 lecturas


,Time,Cars_inside,Total_spaces,Free_spaces,Air_value,Gate_status
0,1900-01-01 11:42:12,0,10,10,39,ALLOW
1,1900-01-01 11:42:13,0,10,10,38,ALLOW
2,1900-01-01 11:42:14,0,10,10,38,ALLOW
3,1900-01-01 11:42:15,0,10,10,38,ALLOW
4,1900-01-01 11:42:16,0,10,10,38,ALLOW


In [5]:
# ── DATOS INVENTADOS POR DÍA DE LA SEMANA (Tab 1) ────────────────────────────
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

# Coches medios por día
coches_dia = [7, 8, 6, 9, 10, 4, 2]

# Entradas y salidas por día
entradas_dia = [14, 16, 12, 18, 20, 8, 4]
salidas_dia  = [13, 15, 12, 17, 19, 8, 4]

# Calidad del aire media por día
aire_dia = [380, 420, 360, 460, 490, 300, 250]

# Porcentaje de tiempo con barrera bloqueada por día
bloqueado_dia = [10, 15, 8, 22, 30, 5, 2]

print("Datos semanales listos")

Datos semanales listos


In [6]:
# ── MÉTRICAS DEL CSV (Tab 2) ──────────────────────────────────────────────────
last     = df.iloc[-1]
cars_now = int(last["Cars_inside"])
free_now = int(last["Free_spaces"])
air_now  = int(last["Air_value"])
gate_now = str(last["Gate_status"]).strip()

diff          = df["Cars_inside"].diff().fillna(0)
total_entries = int((diff > 0).sum())
total_exits   = int((diff < 0).sum())

peak_idx  = df["Cars_inside"].idxmax()
peak_time = df.loc[peak_idx, "Time"].strftime("%H:%M:%S")
peak_cars = int(df.loc[peak_idx, "Cars_inside"])
avg_occ   = round(df["Cars_inside"].mean(), 1)
avg_air   = round(df["Air_value"].mean(), 1)
blocked_pct = round(100 * (df["Gate_status"].str.strip() == "BLOCKED").sum() / len(df), 1)

print(f"Coches ahora: {cars_now} | Libres: {free_now} | Barrera: {gate_now}")

Coches ahora: 8 | Libres: 2 | Barrera: ALLOW
